In [27]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,r2_score
import warnings
warnings.filterwarnings('ignore')   

In [28]:
df = pd.read_csv(r"D:\DA\Archive\GitHUB\Forage-QA\Nat_Gas.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Dates   48 non-null     str    
 1   Prices  48 non-null     float64
dtypes: float64(1), str(1)
memory usage: 900.0 bytes


In [29]:
df.isnull().sum()
df.shape

(48, 2)

In [30]:
df.head(12)



,Dates,Prices
0,10/31/20,10.10
1,11/30/20,10.30
2,12/31/20,11.00
3,1/31/21,10.90
4,2/28/21,10.90
5,3/31/21,10.90
6,4/30/21,10.40
7,5/31/21,9.84
8,6/30/21,10.00
9,7/31/21,10.10


In [31]:
df.isna().sum()



Dates     0
Prices    0
dtype: int64

In [32]:
df['Dates'] = pd.to_datetime(df['Dates'])
df=df.sort_values(by='Dates')
df.head(12)
df.index.name


In [33]:
df.set_index('Dates',inplace=True)
df=df.asfreq('ME')
df.head(12)

,Prices
Dates,
2020-10-31,10.10
2020-11-30,10.30
2020-12-31,11.00
2021-01-31,10.90
2021-02-28,10.90
2021-03-31,10.90
2021-04-30,10.40
2021-05-31,9.84
2021-06-30,10.00


There are no missing or null values in the dataset.

In [34]:
df.shape


(48, 1)

In [35]:
from statsmodels.tsa.seasonal import seasonal_decompose

result=seasonal_decompose(df['Prices'],model='additive',period=12)



the above gives trend,seasonal changes and residual

In [36]:
from statsmodels.tsa.stattools import adfuller

adfuller(df['Prices'])

(np.float64(0.21807686170000193),
 np.float64(0.9732574388448694),
 10,
 37,
 {'1%': np.float64(-3.6209175221605827),
  '5%': np.float64(-2.9435394610388332),
  '10%': np.float64(-2.6104002410518627)},
 np.float64(10.198475035166425))

Data is nor stationary here as p>0.05
So  we need to go with differencing

In [48]:
df['price_diff']=df['Prices'].diff()

df.head()



,Prices,price_diff
Dates,,
2020-10-31,10.1,NaN
2020-11-30,10.3,0.2
2020-12-31,11.0,0.7
2021-01-31,10.9,-0.1
2021-02-28,10.9,0.0


In [49]:
df=df.dropna()

In [50]:
df.shape

(47, 2)

In [51]:
adfuller(df['price_diff'])

(np.float64(-6.844773557477345),
 np.float64(1.754169685294091e-09),
 9,
 37,
 {'1%': np.float64(-3.6209175221605827),
  '5%': np.float64(-2.9435394610388332),
  '10%': np.float64(-2.6104002410518627)},
 np.float64(8.91194325461305))

Data is stationary now , we can apply SARIMA

In [55]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

model=SARIMAX(df['Prices'],order=(1,1,1),seasonal_order=(1,0,1,12))

results=model.fit()

In [61]:
df.head()

,Prices,price_diff
Dates,,
2020-11-30,10.3,0.2
2020-12-31,11.0,0.7
2021-01-31,10.9,-0.1
2021-02-28,10.9,0.0
2021-03-31,10.9,0.0


In [73]:
def predict_gas_price(df, results, input_date):
    import pandas as pd

    input_date = pd.to_datetime(input_date) + pd.offsets.MonthEnd(0)
    last_date = df.index[-1]

    if input_date <= last_date:

        if input_date not in df.index:
            input_date = df.index[df.index.get_indexer([input_date], method='nearest')[0]]

        pred_obj = results.get_prediction(start=input_date, end=input_date)

        pred = pred_obj.predicted_mean.iloc[0]
        conf = pred_obj.conf_int().iloc[0]

    else:
        steps = (input_date.year - last_date.year) * 12 + (input_date.month - last_date.month)

        forecast = results.get_forecast(steps=steps)

        pred = forecast.predicted_mean.iloc[-1]
        conf = forecast.conf_int().iloc[-1]

    return {
        "date": str(input_date),
        "estimated_price": float(pred),
        "confidence_interval": (
            float(conf.iloc[0]),
            float(conf.iloc[1])
        )
    }

In [75]:
result = predict_gas_price(df, results, "2020-06-15")

print(result)

{'date': '2020-11-30 00:00:00', 'estimated_price': 0.0, 'confidence_interval': (-2771.8077516076046, 2771.8077516076046)}
